The aim of this notebook is to explore the data. Handle missing values. Remove redundant features. Return a cleaned dataset ready to evaluate the model on. 

1. Load the data, check types, define the target, drop IDs and leaky columns.
2. Clean: duplicates, types, impossible values, missing values.
3. Summary statistics and categorical value counts.
4. Univariate distributions.
5. Each feature against churn (churn rates, split distributions).
6. Relationships between features (correlation, multicollinearity).
7. Simple feature engineering ideas, documented for the next notebook.
8. Export to parquet

In [ ]:
import pandas as pd 

df = pd.read_csv('../data/raw/telco.csv')
df.shape

In [ ]:
df.isna().sum()

In [ ]:
df.describe(include='all').T

We will use churn label as our main dependent variable for classifier. Secondary classifier could look into churn category, in order to predict reason for churn to customise intervention. 

First there are some very obviously redundant columns 
1. Try and predict churn label, drop customer status 
2. Drop churn score as its not needed
3. ID not useful
4. Country and state and quarter have unique = 1  
5. Latitude and longitude are redundant given we already have city, post code, population

In [ ]:
df = df.drop(columns=['Customer ID', 'Longitude', 'Latitude', 'Churn Score', 'Customer Status', 'Country', 'State', 'Quarter']) 

In [ ]:
df.info(verbose=True, show_counts=True)

In [ ]:
df.select_dtypes(include='str').nunique() 

In [ ]:
df.isna().sum()

Fill nan Internet type with None. As it just means the customer only uses mobile phone plan

In [ ]:
df['Internet Type'].unique()

In [ ]:
df[df['Internet Type'].isna()]['Internet Service'].unique()

In [ ]:
nan_cols = ['Internet Type', 'Offer']
df[nan_cols] = df[nan_cols].fillna('None')

In [ ]:
df.isna().sum()

Further data cleaning: 
1. Take the strs with 'Yes' and 'No' and make a bool, by mapping to true/False
2. Use regex to change 'Offer A' to just A, to simplify. 
4. Change remaining string features with few levels to a category.
5. Rename Churn Label as target

In [ ]:
def yes_no_cols(df):
    cols = [col for col in df.columns if df[col].nunique() == 2 and df[col].dtype == 'str' and any(x in ['Yes', 'No'] for x in df[col].unique().tolist())]
    return df.assign(**{col: df[col].map({'Yes': True, 'No': False}) for col in cols})

In [ ]:
def clean_offer(df): 
    return df.assign(Offer = df['Offer'].str.replace(r'^Offer\s+', '', regex = True))

In [ ]:
def str_to_cat(df):
    cols = [col for col in df.select_dtypes(include='str') if df[col].nunique() <= 6]
    return df.astype({col: 'category' for col in cols})

In [ ]:
df = (
    df.pipe(yes_no_cols)
    .pipe(clean_offer)
    .pipe(str_to_cat)
)

In [ ]:
df = df.rename(columns={'Churn Label': 'target'})

In [ ]:
df.head()

In [ ]:
df[df.duplicated()]

Check for implausible values.

In [ ]:
df.describe()

In [ ]:
df.info(verbose=True, show_counts=True)

## EDA

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import scipy.stats 

I want to understand what features are related to internet versus phone service. 
- Some numeric features are 0, this means that they may not be using this feature. 
- Some categorical features are set to None ('Internet Type').
- Some booleans are set to False, is this linked to service used. 

In [ ]:
service_cols = [
    "Multiple Lines", "Avg Monthly Long Distance Charges", "Total Long Distance Charges",
    "Internet Type", "Avg Monthly GB Download", "Total Extra Data Charges",
    "Online Security", "Online Backup", "Device Protection Plan", "Premium Tech Support",
    "Streaming TV", "Streaming Movies", "Streaming Music", "Unlimited Data",
]

In [ ]:
def uses(s): 
    if s.dtype == 'bool': 
        return s 
    if isinstance(s.dtype, pd.CategoricalDtype): 
        return s.ne("None")
    return s.gt(0)

In [ ]:
condlist = [(df['Phone Service'] == True) & (df['Internet Service'] == False), 
            (df['Phone Service'] == False) & (df['Internet Service'] == True),
            (df['Phone Service'] == True) & (df['Internet Service'] == True)]

df['Segment'] = np.select(condlist=condlist, choicelist=['Phone Only', 'Internet Only', 'Both Service'], default='Neither') 

In [ ]:
df['Segment'].value_counts() / len(df) * 100

In [ ]:
df.groupby("target")["Segment"].value_counts(normalize = True)

In [ ]:
usage = df[service_cols].apply(uses).groupby(df["Segment"]).mean().T

sns.heatmap(usage, annot=True, fmt=".0%", cmap="Blues", vmin=0, vmax=1)

In [ ]:
phone_add_ons = ['Multiple Lines']
internet_add_ons = ['Online Security', 'Online Backup', 'Device Protection Plan', 'Premium Tech Support', 
                    'Streaming TV', 'Streaming Movies', 'Streaming Music', 'Unlimited Data']

In [ ]:
df.query("`Phone Service` == True | `Segment` == 'Both Service'").groupby('target')['Multiple Lines'].value_counts(normalize = True)

In [ ]:
internet_users = df.query("`Internet Service` | `Segment` == 'Both Service'")

long = internet_users.melt(
    id_vars="target", value_vars=internet_add_ons,
    var_name="add_on", value_name="has_add_on",
)

plt.figure(figsize=(12, 5))
sns.barplot(data=long, x="add_on", y="target", hue="has_add_on")
plt.axhline(internet_users["target"].mean(), ls="--", color="grey")
plt.ylabel("Churn rate")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

First, I want look at the numeric variables and see what the median values are for churners versus not churners. This table tells me a few things to look in driving churn / linked to churn. 
- There are obvious things that indicate a customer may be less likely to churn, number of referrels and tenure in months, these customers reccommend the service and have been using it for longer. I'd argue the longer you use a service the less likely you are to switch, this should be checked further. 
- Churners are charged more a month, and get less in refunds. 
- Non churners are as expected more satisfied with the service. 

**Modelling Considerations**: The satisfaction score cuts churners and non-churners very cleanly splits the two, to the point that I think this is data leakage. We won't use this feature to train the models as in reality, services like this don't always have live satisfaction scores from customers. Same with CLTV, not sure how this was obtained, we will not use in modelling. 

In [ ]:
num_cols = df.select_dtypes(include='number').columns.to_list() + ['target']
df[num_cols].groupby('target').aggregate(["median", "mean"]).round(2).T

In [ ]:
df.groupby('target')['Satisfaction Score'].aggregate(["mean", "median", "max", "min"])

And to get an understanding of the variability, 

In [ ]:
df[num_cols].groupby('target').agg(["std", scipy.stats.median_abs_deviation]).round(2).T

Looking for features which have a large difference in median/mean for churners against non-churners, and especially those features with less variability in churners. 

In [ ]:
cols = ["Number of Referrals", "Tenure in Months", "Avg Monthly GB Download", "Monthly Charge", "Total Long Distance Charges"]

In [ ]:
fig, axes = plt.subplots(len(cols), 2, figsize = (12, 5 * len(cols)))

for (ax_hist, ax_box), col in zip(axes, cols): 
    sns.histplot(data = df, x = col, hue = 'target', stat = 'density', ax = ax_hist, common_norm=False, 
                 binrange=(df[col].min(), df[col].quantile(0.99)), multiple='stack')
    sns.boxplot(data = df, x = 'target', y = col, ax = ax_box, showfliers = False)

    ax_hist.set_title(col)

plt.show()

Now let's look at the categorical features. What is the distribution of churners in each catgeory/bool?

In [ ]:
bool_cat_cols = df.select_dtypes(include = ['category', 'bool']).columns.to_list()
bool_cat_cols.remove('Churn Category')

In [ ]:
churn_dist = (
    df.query("target")[bool_cat_cols]
    .drop(columns = 'target')
    .melt(var_name = 'feature', value_name = "value")
    .groupby("feature")["value"]
    .value_counts(normalize = True)
)

This is valuable,
- 88% of churners are on month-to-month contract
- Churners usually have no dependents 
- 94% use the internet service
- 66% who use internet service who churn use the fibre obtic
- Offer E has large churn amt amongst all those who have offer and churn (follow up, whats special about offer E?)
- Churners tend to not have premium tech support

In [ ]:
pd.DataFrame(churn_dist.T)

In [ ]:
pd.crosstab(df['Offer'], df['Contract'], normalize='index')

In [ ]:
pd.crosstab(df['Offer'], df['Contract'], normalize='columns')

In [ ]:
bool_cols = df.select_dtypes(include='bool').columns.to_list()

In [ ]:
df.groupby('Offer').agg({col: ['median'] for col in num_cols} | {col: ["mean"] for col in bool_cols}).round(2).T

In [ ]:
df.groupby('Offer')['Tenure in Months'].agg(['min', 'median', 'max'])

How many customers churned this quarter? How much total revenue will be lost for following quarter?

In [ ]:
df['target'].value_counts() / len(df) * 100

In [ ]:
df.groupby('target')['Total Revenue'].sum()

Looking at churn rate.

In [ ]:
bool_cols = df.select_dtypes(include = 'bool').columns.tolist()
overall_churn = df['target'].mean()

# Group by boolean columns and look at what % churned vs didnt 
churn_rates = pd.concat({col: df.groupby(col, observed = True)['target'].agg(churn_rate = 'mean' ,n = 'size') for col in bool_cols}, names = ['feature', 'value'])

In [ ]:
churn_rates['lift'] = churn_rates['churn_rate'] / overall_churn

Observations 
- Senior citizens 50% more likely on average to churn. Higher median age for churners. 41% of senior citizens were churners > overall average of 26%. 
- Some characteristics that seem 
- As expected lower median tenure for churners, lower revenue and LD charges
- Lower satisfaction score
- Churners tended to download more, and had higher total charges 
- Not referring a friend makes someone more likely to churn
- Not having dependents/being single, also more likely to churn. 

In [ ]:
churn_rates.sort_values(by='lift', ascending = False)